In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (8693, 14)
Test shape: (4277, 13)


In [3]:
y = train["Transported"]

X = train.drop(["Transported"], axis=1)
X_test = test.copy()

combined = pd.concat([X, X_test], axis=0).reset_index(drop=True)

In [4]:
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
for col in num_cols:
    combined[col] = combined[col].fillna(combined[col].median())

cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
for col in cat_cols:
    combined[col] = combined[col].fillna(combined[col].mode()[0])

combined["Cabin"] = combined["Cabin"].fillna("U/0/U")

In [5]:
combined["CabinSide"] = combined["Cabin"].apply(lambda x: str(x).split("/")[-1])

In [6]:
columns_to_drop = ["PassengerId", "Cabin", "Name"]
combined = combined.drop(columns_to_drop, axis=1)

combined = pd.get_dummies(combined, columns=["HomePlanet", "Destination", "CryoSleep", "VIP", "CabinSide"], drop_first=True)

X_train = combined.iloc[:len(train)].copy()
X_test = combined.iloc[len(train):].copy()


In [7]:
X_local_train, X_local_val, y_local_train, y_local_val = train_test_split(
    X_train, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

model.fit(X_local_train, y_local_train)

predictions = model.predict(X_local_val)
accuracy = accuracy_score(y_local_val, predictions)

print("Our local validation accuracy is: {accuracy * 100:.2f}%")

Our local validation accuracy is: {accuracy * 100:.2f}%


In [8]:
model.fit(X_train, y)

final_predictions = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId":test["PassengerId"],
    "Transported":final_predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv is successfully made and ready to submit!")

submission.csv is successfully made and ready to submit!
